# 01 - Test de l'infrastructure
Objectif : vérifier que Spark (1 master, 2 workers) écrit et relit une table Delta sur HDFS, et que le MERGE fonctionne.


## 1.1  ouvrir la spark

In [1]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

spark = SparkSession.builder.appName("01_infrastructure").getOrCreate()

print("Version Spark      :", spark.version)
print("Master             :", spark.sparkContext.master)
print("Stockage par défaut:", spark.conf.get("spark.hadoop.fs.defaultFS"))



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 13:39:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Version Spark      : 3.5.7
Master             : spark://spark-master:7077
Stockage par défaut: hdfs://namenode:9000
Cœurs disponibles  : 2


In [2]:
print("Cœurs disponibles  :", spark.sparkContext.defaultParallelism)

Cœurs disponibles  : 4


## 1.2  ecriture  et lecture

In [3]:
CHEMIN = "/test/delta_test"   # sur HDFS, car fs.defaultFS = hdfs://namenode:9000

depart = spark.createDataFrame(
    [("s1", "automatic", 4.6), ("s2", "automatic", 5.1)],
    ["id", "status", "mag"],
)
depart.write.format("delta").mode("overwrite").save(CHEMIN)

spark.read.format("delta").load(CHEMIN).orderBy("id").show()


26/11/12 13:42:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---+---------+---+
| id|   status|mag|
+---+---------+---+
| s1|automatic|4.6|
| s2|automatic|5.1|
+---+---------+---+



In [ ]:
1.